# Spot 모델 검증

두 세트의 URDF/MJCF가 서로 매칭되는지 + MuJoCo가 정상 스폰되는지 확인.

| 세트 | URDF (pinocchio) | MJCF (MuJoCo) |
|---|---|---|
| body only | `urdf/spot_simple.urdf` | `xml/spot.xml` |
| body + arm + gripper | `urdf/spot_arm.urdf` | `xml/spot_arm.xml` |

In [ ]:
import sys
from pathlib import Path

_HERE = Path.cwd() if '__file__' not in globals() else Path(__file__).resolve().parent
_ROOT = _HERE.parents[0]
sys.path.insert(0, str(_HERE))

import numpy as np
import pinocchio as pin
from pinocchio.robot_wrapper import RobotWrapper
import mujoco

SPOT = str(_ROOT / 'models' / 'spot')

## 1. 모델 로딩 (pinocchio + MuJoCo, 두 세트)

In [ ]:
def load_pair(urdf_name, xml_name):
    urdf = f'{SPOT}/urdf/{urdf_name}'
    xml  = f'{SPOT}/xml/{xml_name}'
    pin_robot = RobotWrapper.BuildFromURDF(urdf, package_dirs=[SPOT],
                                            root_joint=pin.JointModelFreeFlyer())
    mj_model  = mujoco.MjModel.from_xml_path(xml)
    mj_data   = mujoco.MjData(mj_model)
    return pin_robot, mj_model, mj_data

pin_body, mj_body, mjd_body = load_pair('spot_simple.urdf', 'spot.xml')
pin_arm,  mj_arm,  mjd_arm  = load_pair('spot_arm.urdf',    'spot_arm.xml')

print('body only 로딩 OK')
print('  pinocchio: nq=%d nv=%d nframes=%d' % (pin_body.model.nq, pin_body.model.nv, pin_body.model.nframes))
print('  mujoco   : nq=%d nv=%d nbody=%d'   % (mj_body.nq, mj_body.nv, mj_body.nbody))
print()
print('body+arm+gripper 로딩 OK')
print('  pinocchio: nq=%d nv=%d nframes=%d' % (pin_arm.model.nq, pin_arm.model.nv, pin_arm.model.nframes))
print('  mujoco   : nq=%d nv=%d nbody=%d'   % (mj_arm.nq, mj_arm.nv, mj_arm.nbody))

## 2. Joint 이름·개수 매칭

In [ ]:
def compare_joints(name, pin_robot, mj_model):
    # pinocchio: universe(0), freeflyer(1) 제외하고 실제 joint만
    pin_joints = [pin_robot.model.names[i] for i in range(2, pin_robot.model.njoints)]
    # mujoco: free joint은 종종 없음 (MJCF가 root_joint 안 만들면 base는 worldbody에 직접 붙음)
    mj_joints = [mj_model.joint(i).name for i in range(mj_model.njnt)]
    mj_joints_no_free = [j for j in mj_joints if 'free' not in j.lower()]
    print(f'\n=== {name} ===')
    print(f'pinocchio joints: {len(pin_joints)}')
    print(f'  {pin_joints}')
    print(f'mujoco joints (free 제외): {len(mj_joints_no_free)}')
    print(f'  {mj_joints_no_free}')
    # 매칭
    pin_set = set(pin_joints)
    mj_set  = set(mj_joints_no_free)
    common  = pin_set & mj_set
    print(f'공통 joint: {len(common)} / pin {len(pin_set)} / mj {len(mj_set)}')
    only_pin = pin_set - mj_set
    only_mj  = mj_set - pin_set
    if only_pin: print(f'  pin only: {sorted(only_pin)}')
    if only_mj : print(f'  mj  only: {sorted(only_mj)}')

compare_joints('body only',        pin_body, mj_body)
compare_joints('body+arm+gripper', pin_arm,  mj_arm)

## 3. Total mass / Center of mass 비교

In [ ]:
def compare_mass_com(name, pin_robot, mj_model, mj_data):
    # pinocchio
    m_pin = pin.computeTotalMass(pin_robot.model)
    q_neutral = pin.neutral(pin_robot.model)
    com_pin = pin.centerOfMass(pin_robot.model, pin_robot.data, q_neutral)
    # mujoco — keyframe 없으면 zero qpos, 있으면 keyframe[0]
    if mj_model.nkey > 0:
        mujoco.mj_resetDataKeyframe(mj_model, mj_data, 0)
    else:
        mujoco.mj_resetData(mj_model, mj_data)
    mujoco.mj_forward(mj_model, mj_data)
    m_mj  = mj_model.body_mass.sum()
    com_mj = mj_data.subtree_com[0]
    print(f'\n=== {name} ===')
    print(f'total mass  pinocchio: {m_pin:.4f} kg')
    print(f'total mass  mujoco   : {m_mj:.4f} kg   (diff: {m_pin - m_mj:+.4f})')
    print(f'CoM (neutral) pinocchio: {np.round(com_pin, 4)}')
    print(f'CoM (keyframe) mujoco  : {np.round(com_mj, 4)}')

compare_mass_com('body only',        pin_body, mj_body, mjd_body)
compare_mass_com('body+arm+gripper', pin_arm,  mj_arm,  mjd_arm)

## 4. MuJoCo viewer 스폰 (body only)

viewer가 뜨고 spot 모델이 floor 위에 정상적으로 보이면 OK.
viewer 창은 직접 닫으면 다음 셀로 진행.

In [ ]:
import mujoco.viewer

scene = mujoco.MjModel.from_xml_path(f'{SPOT}/xml/scene.xml')
data  = mujoco.MjData(scene)
if scene.nkey > 0:
    mujoco.mj_resetDataKeyframe(scene, data, 0)

with mujoco.viewer.launch_passive(scene, data) as viewer:
    while viewer.is_running():
        mujoco.mj_step(scene, data)
        viewer.sync()

## 5. MuJoCo viewer 스폰 (body + arm)

In [ ]:
scene_arm = mujoco.MjModel.from_xml_path(f'{SPOT}/xml/scene_arm.xml')
data_arm  = mujoco.MjData(scene_arm)
if scene_arm.nkey > 0:
    mujoco.mj_resetDataKeyframe(scene_arm, data_arm, 0)

with mujoco.viewer.launch_passive(scene_arm, data_arm) as viewer:
    while viewer.is_running():
        mujoco.mj_step(scene_arm, data_arm)
        viewer.sync()